# Code demo with OpenAI API

The current jupyter notebook demonstrates the workflow of the benchmark using OpenAI API as an example. This notebook will also breakdown core components and functions used in this repository, including loading task data, running models, and evaluating responses.

In [ ]:
import json
import os

from dotenv import load_dotenv

from neuropsych_bench import Evaluator, ModelConfig, OpenAIModelRunner, TaskLoader, build_parts, load_rows, task_names

## Load API Keys

API keys are read from a `.env` file in the repository root (see `.env.example`):

```
OPENAI_KEY=your_openai_api_key
ANTHROPIC_KEY=your_anthropic_api_key
GEMINI_KEY=your_google_api_key
```

In [ ]:
load_dotenv()

## Inspect Available Tests

The code below shows a sample of the tests in the benchmark.
In the paper: https://arxiv.org/abs/2504.10786v1. Neuropsychological tests are categorized into which visual processing stage they are designed to tap into- low, mid, or high.

The whole benchmark is one file, `data/neuropsych_bench.jsonl`, with one line per trial (see `data/README.md`). Each line holds the task groupings, task name, task type, the prompt, the trial's images in presentation order (paths relative to `data/`), the role of each image (target or option), and the answer key.

In [ ]:
all_tasks = task_names()
print(f"Found {len(all_tasks)} tasks")
print("Sample tasks:")
for i, task in enumerate(all_tasks[:3]):
    print(f"  {i}: {task}")

In [ ]:
# Show the fields of one trial
rows = load_rows()
print(json.dumps(rows[0], indent=2))

## Pipeline Demo with OpenAI API

The following will walk you through the process of running and evaluating a single test using OpenAI API.

The pipeline involves the following core components:

1. **TaskLoader**: Loads the trials of one task from `data/neuropsych_bench.jsonl`.
2. **build_parts** (`neuropsych_bench/prompting.py`): Turns a trial into the ordered text and image parts the model sees. Every model runner sends exactly these parts, so the prompt format is the same across providers and consistent with the original paper.
3. **OpenAIModelRunner**: This is a wrapper around the OpenAI API and handles the API calls.
4. **Evaluator**: Evaluates the model responses against the answer keys.

### Loading a task

TaskLoader is initialized with a task name:

```python
loader = TaskLoader("borb_line_length_comparison")
```

The function `loader.get_task_info()` returns a dictionary containing the task information.

```python
loader.get_task_info()
```

While `loader.get_trials()` returns a list of dictionaries containing the trial-by-trial information.

```python
loader.get_trials()
```

`build_parts(trial, trial["data_root"])` shows exactly what the model will be sent for a trial.

In [ ]:
# Load one of the tests
task_index = 0
print(f"We will run task: {all_tasks[task_index]}")

## LOAD TASK
loader = TaskLoader(all_tasks[task_index])
print(loader.get_task_info())

# What the model is sent for the first trial: the prompt, then the images (with cues in three- and four-image trials)
trial = loader.get_trials()[0]
build_parts(trial, trial["data_root"])

### Initialize the Model Runner and Run the Test

ModelConfig is a dataclass that stores the configuration for the model runner. It contains the model name, max tokens, temperature, api key, and any additional parameters. This is used to initialize the model runner.

We have three model runners already available: OpenAIModelRunner, AnthropicModelRunner, and GoogleModelRunner. These are the providers of models used in the original paper. If you want to use a different model, you can create a new model runner by inheriting from the ModelRunner class.

Once the model runner is initialized, we can use it to run the tests. The model runner's function generate_response() takes a TaskLoader object as input and returns a tuple containing the task information and the model responses. This consists of the task information originally existing in the test metadata, but now with the addition of the model's responses for each trial.

In [ ]:
## INITIALIZE
# Configure OpenAI model
openai_config = ModelConfig(
    model_name="gpt-6-sol", # model name (the paper used gpt-4o-2024-05-13, now retired)
    api_key=os.getenv("OPENAI_KEY"), # set in .env
    max_tokens=1000, # max tokens used to generate the response
    temperature=None, # None = vendor default temperature (the paper used 1.0)
    additional_params={"reasoning_effort": "none"} # non-reasoning mode
)

# Initialize runner
runner = OpenAIModelRunner(openai_config)

## RUN THE TEST
# Run the task
results = runner.generate_response(loader) # return task info and results (payload + model responses)

# Here `results` is a tuple containing the task info and the results
print(f"Task: {results[0]['task']}")
print(f"Stage: {results[0]['stage']}") # stage of the task: low, mid, or high
print(f"Process: {results[0]['process']}") # which finer-grained process the task belongs to
print(f"Number of trials: {len(results[1])}") # number of trials in the task

# Print example model response
print(results[1][0]['model_response']) #[1] returns trial-by-trial data, [0] reflects the first trial

In [ ]:
# Show example of trial-by-trial details

# Show first few responses
print("Sample Model Responses:")
print("=" * 50)

# Look at the results
for i, trial in enumerate(results[1][:3]):
    print(f"\nTrial {trial['trial_id']}:")
    print(f"Prompt: {trial['prompt'][:100]}...") # Prompts are same for all the trials within a task.
    print(f"Model Response: {trial['model_response']}") # Final model responses are those indicated within {}
    print(f"Correct Answer: {trial['answer_key']}")
    print("-" * 30)

### Evaluate the Results

The current repository provides a class that helps automatically mark the model responses based on the criteria used in the paper.

The evaluator takes the output tuple of the runner that contains both the overall task information and trial-by-trial information. The evaluator reads evaluation config file from `neuropsych_bench/resources/evaluator_config.json`* to determine the method of evaluation for each task.

Inputting the evaluator with the output will initialize marking process. The results are stored as a table and can be accessed using `.get_results()`.

The results table can be saved as a CSV file using `.save_as_csv()`. Its argument is the path of the file.

For every task evaluated the evaluator will store the corresponding result as a row in the table. Running and evaluating through all tasks will result in a table with the results for all tasks. *See below for how to run a batch of tasks*.

<br>

*`evaluator_config.json` is a file that contains the list of tasks that use each evaluation method. 

In [ ]:
# Evaluate results
evaluator = Evaluator()
evaluator.evaluate(results)

# Display results
results_df = evaluator.get_result() # get pandas dataframe
results_df

### Running a Batch of Tasks

We can run loop through multiple tasks and input the results into the evaluator.

In [ ]:
# Run a subset of tasks
n_tasks = 5

# Initialize evaluator
batch_evaluator = Evaluator()
runner = OpenAIModelRunner(openai_config) # Initialize the runner

# Loop through the tasks
for i, task in enumerate(all_tasks[:n_tasks]):
    print(f"
Processing task {i+1}/{n_tasks}: {task}")

    loader = TaskLoader(task) # Load the task data
    results = runner.generate_response(loader) # Get the model responses
    batch_evaluator.evaluate(results) # Evaluate the result, this will append to the result table stored
    print(f"Completed: {results[0]['task']}")

# Save results
batch_evaluator.save_as_csv(f"results_{openai_config.model_name}.csv")
batch_evaluator.save_trials_as_csv(f"trials_{openai_config.model_name}.csv") # trial-by-trial responses and correctness